In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import zipfile

with zipfile.ZipFile('/Potato_plant.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

In [5]:
import os
import cv2
import numpy as np

IMG_SIZE = 64

classes = [
    "Potato___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight"
]

data = []
labels = []

for label, cls in enumerate(classes):
    folder = os.path.join('/content', cls)

    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        data.append(img)
        labels.append(label)

X = np.array(data) / 255.0
y = np.array(labels)

# Flatten
X = X.reshape(X.shape[0], -1)

print("Total dataset:", X.shape)

Total dataset: (2152, 12288)


In [6]:
from sklearn.model_selection import train_test_split

X_train_full, X_test_global, y_train_full, y_test_global = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train_full.shape)
print("Global Test:", X_test_global.shape)

Train: (1721, 12288)
Global Test: (431, 12288)


In [7]:
num_clients = 3
client_data = []

split_size = len(X_train_full) // num_clients

for i in range(num_clients):
    start = i * split_size
    end = (i + 1) * split_size if i != num_clients - 1 else len(X_train_full)

    X_c = X_train_full[start:end]
    y_c = y_train_full[start:end]

    client_data.append((X_c, y_c))

print("Clients created:", len(client_data))

Clients created: 3


In [8]:
import numpy as np

# Clean all client data
client_data_fixed = []

for X, y in client_data:
    idx = np.random.permutation(len(X))
    X = X[idx]
    y = y[idx]
    client_data_fixed.append((X, y))

client_data = client_data_fixed

# Clean test data
X_test_global = np.nan_to_num(X_test_global)

In [9]:
def quantum_transform(X):
    return np.concatenate([X, np.cos(np.pi * X)], axis=1)
client_data = [(quantum_transform(X), y) for X, y in client_data]
X_test_global = quantum_transform(X_test_global)

In [10]:
from sklearn.linear_model import SGDClassifier

def train_local_model(X, y, epochs=7, batch_size=32):

    model = SGDClassifier(
        loss='log_loss',
        learning_rate='adaptive',
        eta0=0.001,
        max_iter=1,
        tol=None
    )

    n = len(X)

    for epoch in range(epochs):
        for i in range(0, n, batch_size):
            X_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]

            if epoch == 0 and i == 0:
                model.partial_fit(X_batch, y_batch, classes=np.unique(y))
            else:
                model.partial_fit(X_batch, y_batch)

    return model

In [11]:
def fed_avg(models):
    avg_coef = np.mean([m.coef_ for m in models], axis=0)
    avg_intercept = np.mean([m.intercept_ for m in models], axis=0)
    return avg_coef, avg_intercept

In [12]:
global_model=SGDClassifier(
    loss='log_loss',
    learning_rate='adaptive',
    eta0=0.001,
    class_weight='balanced'
)

In [13]:
from sklearn.metrics import log_loss

max_rounds = 100
patience = 8

best_loss = float('inf')
no_improve = 0

for round_num in range(max_rounds):

    print(f"\n--- Round {round_num+1} ---")

    client_models = []

    #  Train each client
    for X_c, y_c in client_data:
        model = train_local_model(X_c, y_c)
        client_models.append(model)

    # Federated Averaging
    avg_coef, avg_intercept = fed_avg(client_models)

    global_model.coef_ = avg_coef
    global_model.intercept_ = avg_intercept
    global_model.classes_ = np.unique(y_train_full)

    # Evaluate safely
    y_prob = global_model.predict_proba(X_test_global)
    y_prob = np.nan_to_num(y_prob)   # VERY IMPORTANT FIX

    loss = log_loss(y_test_global, y_prob)

    print("Loss:", loss)

    # Early stopping
    if loss < best_loss:
        best_loss = loss
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= patience:
        print("Converged. Stopping.")
        break


--- Round 1 ---
Loss: 0.2781003512909463

--- Round 2 ---
Loss: 0.2818047495661504

--- Round 3 ---
Loss: 0.2835612795312008

--- Round 4 ---
Loss: 0.29310766175197367

--- Round 5 ---
Loss: 0.27934522235787995

--- Round 6 ---
Loss: 0.2769603449164862

--- Round 7 ---
Loss: 0.2821896890648473

--- Round 8 ---
Loss: 0.27410255975198244

--- Round 9 ---
Loss: 0.28344690027503594

--- Round 10 ---
Loss: 0.2858484076115844

--- Round 11 ---
Loss: 0.278859295903918

--- Round 12 ---
Loss: 0.2883874292195175

--- Round 13 ---
Loss: 0.28717180885641574

--- Round 14 ---
Loss: 0.28450303277380085

--- Round 15 ---
Loss: 0.2850502753132174

--- Round 16 ---
Loss: 0.2764697273459779
Converged. Stopping.


In [14]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

y_pred = global_model.predict(X_test_global)

print("\nFinal Results:")
print("Accuracy:", accuracy_score(y_test_global, y_pred))
print("F1 Score:", f1_score(y_test_global, y_pred, average='weighted'))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_global, y_pred))


Final Results:
Accuracy: 0.8909512761020881
F1 Score: 0.8822924676633145

Confusion Matrix:
[[ 11   1  19]
 [  0 192   8]
 [  2  17 181]]


In [28]:
import cv2
import numpy as np
 # Load image
img = cv2.imread("sample_data/test_leaf5.jpg")
 # Preprocess
img = cv2.resize(img, (64, 64))
img = img / 255.0
img = img.reshape(1, -1)
 # Quantum transform
img = np.concatenate([img, np.cos(np.pi * img)], axis=1)
  # Predict
pred = global_model.predict(img)
  # Map label
labels = ["Healthy", "Early Blight", "Late Blight"]
print("Prediction:", labels[pred[0]])

Prediction: Late Blight
